In [1]:
import pyomo.environ as pyo
import pandas as pd
import math
import numpy as np
from collections import defaultdict
from datetime import timedelta
from pyomo.util.infeasible import log_infeasible_constraints
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
from matplotlib import cm
from pyomo.opt import TerminationCondition
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
class charging_point():
    def __init__(self, name, ev_id, session_id, ev_capacity, ev_max_power, ev_arrival_soc, ev_arrival, ev_departure, ev_desired_soc):
        self.efficiency = 0.93
        # Input validation using assertions
        assert isinstance(ev_capacity, list), "ev_capacity must be a list"
        assert isinstance(ev_max_power, list), "ev_max_power must be a list"
        assert isinstance(ev_arrival_soc, list), "ev_arrival_soc must be a list"
        assert isinstance(ev_arrival, list), "ev_arrival must be a list"
        assert isinstance(ev_departure, list), "ev_departure must be a list"
        assert isinstance(ev_desired_soc, list), "ev_desired_soc must be a list"

        assert len(ev_arrival) == len(ev_departure), "ev_arrival and ev_departure must have the same length"
        assert len(ev_arrival) == len(ev_desired_soc), "ev_arrival and ev_desired_soc must have the same length"
        assert len(ev_arrival_soc) == len(ev_desired_soc), "ev_arrival_soc and ev_desired_soc must have the same length"
        assert len(ev_capacity) == len(ev_desired_soc), "ev_capacity and ev_desired_soc must have the same length"
        assert len(ev_max_power) == len(ev_desired_soc), "ev_max_power and ev_desired_soc must have the same length"

        for i in range(len(ev_arrival)):
            assert ev_arrival[i] < ev_departure[i], f"ev_arrival[{i}] must be less than ev_departure[{i}]"
            assert 0.2 <= ev_arrival_soc[i] <= 1, f"ev_arrival_soc[{i}] must be between 0.2 and 1"
            assert 0 <= ev_desired_soc[i] <= 1, f"ev_desired_soc[{i}] must be between 0 and 1"
            min_time_to_charge = ev_departure[i] - ev_arrival[i]
            min_req_charge = (ev_desired_soc[i] - ev_arrival_soc[i]) * ev_capacity[i] / self.efficiency
            min_req_charge_per_time = min_req_charge / min_time_to_charge
            assert min_req_charge_per_time <= ev_max_power[i], f"min_req_charge_per_time ({min_req_charge_per_time:.2f}) must be less than or equal to ev_max_power[{i}] ({ev_max_power[i]}) for {name}"
            if min_req_charge_per_time * 4 >= ev_max_power[i]:
                warnings.warn(f"Charging would not work for 15-min resolution for {name}")


        self.name = name
        self.ev_capacity = ev_capacity
        self.ev_max_power = ev_max_power
        self.ev_arrival_soc = ev_arrival_soc
        self.ev_desired_soc = ev_desired_soc
        self.ev_arrival = ev_arrival
        self.ev_departure = ev_departure
        self.num_evs = len(ev_capacity) # Store the number of EVs
        self.ev_id = ev_id
        self.session_id = session_id
        self.ch_max_power = max(ev_max_power)
        self.ds_max_power = max(ev_max_power)

class building():
    def __init__(self, name, load, pv_production, bess_capacity, bess_max_power, bess_initial_soc, heating_load = None, has_hp = False, hp_cop = 3.0, hp_max_power = 0, 
                 cooling_load = None, has_ch = False, ch_cop = 2.0, ch_max_power = 0, ch_heat_rec_eff = 0, has_abs_ch = False, abs_ch_cop = 2.0, abs_max_heat = 0,
                 el_imp_max = 5000, el_exp_max = 5000, el_int_ex_max = 500, heat_int_ex_max = 500, cool_int_ex_max = 500, dc_imp_max = 100000):
        self.efficiency = 0.93
        self.name = name
        self.load = load
        #self.cooling_load = self.cooling_load
        if heating_load is None:
            self.heating_load = [0] * len(load)
        else:
            assert len(heating_load) == len(load), "Heating load must match length of load"
            self.heating_load = heating_load
        self.has_hp = has_hp
        if self.has_hp:
            self.hp_cop = hp_cop
        else:
            self.hp_cop = 0
        self.hp_max_power = hp_max_power
        if cooling_load is None:
            self.cooling_load = [0] * len(load)
        else:
            assert len(cooling_load) == len(load), "Cooling load must match length of load"
            self.cooling_load = cooling_load
        self.has_ch = has_ch
        if self.has_ch:
            self.ch_cop = ch_cop
        else:
            self.ch_cop = 0
        self.has_abs_ch = has_abs_ch
        if self.has_abs_ch:
            self.abs_ch_cop = abs_ch_cop
        else:
            self.abs_ch_cop = 0
        self.abs_max_heat = abs_max_heat
        self.ch_max_power = ch_max_power
        self.ch_heat_rec_eff = ch_heat_rec_eff
        self.pv_production = pv_production
        self.bess_capacity = bess_capacity
        self.bess_max_power = bess_max_power
        self.bess_initial_soc = bess_initial_soc
        self.el_imp_max = el_imp_max
        self.el_exp_max = el_exp_max
        self.el_int_ex_max = el_int_ex_max
        self.heat_int_ex_max = heat_int_ex_max
        self.cool_int_ex_max = cool_int_ex_max
        self.dc_imp_max = dc_imp_max

class LEC_Opt_spot_multi_energy():
    def __init__(self, charging_points, buildings, spot_prices, dh_prices, dc_prices, previous_monthly_peak=0, v2g_on=1, resolution = 1, dc = False, subscription_fee = 605/30, subscription_fee_dh = 117820/365,
                  subscription_fee_dc = 80020/365, transmission_fee = 0.113, peak_effect_fee_dh = 1185/365, peak_effect_fee_dc = 695/365,
                 transmission_health_incentive = 0.04, peak_effect_fee = 61.55/30, energy_tax = 0.439, energy_certificate = 0.005, incentive_per_kwh=0.1, previous_monthly_peak_dc = 0,
                 el_import_limit = 100000, dh_import_limit = 10000, dc_import_limit = 10000, previous_monthly_peak_dh = 0):
        self.M = 10000
        self.charging_points = charging_points
        self.buildings = buildings
        self.spot_prices = spot_prices
        self.dh_prices = dh_prices
        self.dc_prices = dc_prices
        self.incentive_per_kwh = incentive_per_kwh
        self.v2g_on = v2g_on
        self.previous_monthly_peak = previous_monthly_peak
        self.resolution = resolution
        self.dc = dc
        self.subscription_fee = subscription_fee
        self.subscription_fee_dh = subscription_fee_dh
        self.subscription_fee_dc = subscription_fee_dc
        self.tranmission_fee = transmission_fee
        self.tranmission_health_incentive = transmission_health_incentive
        self.peak_effect_fee = peak_effect_fee
        self.peak_effect_fee_dh = peak_effect_fee_dh
        self.peak_effect_fee_dc = peak_effect_fee_dc
        self.energy_tax = energy_tax
        self.energy_certificate = energy_certificate
        self.el_import_limit = el_import_limit
        self.dh_import_limit = dh_import_limit
        self.dc_import_limit = dc_import_limit
        self.previous_monthly_peak_dc = previous_monthly_peak_dc
        self.previous_monthly_peak_dh = previous_monthly_peak_dh
        self.model = pyo.ConcreteModel()
        self.build_model()

    def build_model(self):
        self.model.T = pyo.Set(initialize=range(len(self.spot_prices)))
        self.model.spot_prices = self.spot_prices
        self.model.dh_prices = self.dh_prices 
        self.model.dc_prices = self.dc_prices
        self.model.previous_monthly_peak = self.previous_monthly_peak
        self.model.previous_monthly_peak_dc = self.previous_monthly_peak_dc
        self.model.previous_monthly_peak_dh = self.previous_monthly_peak_dh
        self.model.monthly_peak = pyo.Var(initialize = 0)
        self.model.monthly_peak_dc = pyo.Var(initialize = 0)
        self.model.monthly_peak_dh = pyo.Var(initialize = 0)
        self.model.dh_import = pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0, self.dh_import_limit))
        self.model.dc_import = pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0, self.dc_import_limit))
        self.model.P_im_grid = pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, self.el_import_limit))
        self.model.P_ex_grid = pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, 100000))
        self.model.B_im_grid = pyo.Var(self.model.T, within=pyo.Binary)
        self.model.Peakload = pyo.Var(within=pyo.NonNegativeReals)
        self.model.Peakload_dc = pyo.Var(within = pyo.NonNegativeReals)
        self.model.Peakload_dh = pyo.Var(within = pyo.NonNegativeReals)
        self.model.transmission_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.supplier_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.dh_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.dc_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.overall_dso_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.tax_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.peak_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.Subscription_fee = self.subscription_fee  # Subscription fee SEK
        self.model.Subscription_fee_dc = self.subscription_fee_dc  # Subscription fee SEK
        self.model.Subscription_fee_dh = self.subscription_fee_dh  # Subscription fee SEK
        self.model.Transmission_fee = self.tranmission_fee  # Electricity transmission fee SEK/kWh
        self.model.Transmission_health_incentive = self.tranmission_health_incentive #Transmission health incentive SEK/kWh
        self.model.Effect_fee = self.peak_effect_fee       # Effect fee SEK/kW
        self.model.Effect_fee_dc = self.peak_effect_fee_dc
        self.model.Effect_fee_dh = self.peak_effect_fee_dh
        self.model.Energy_tax = self.energy_tax           # Tax fee SEK/kWh
        self.model.Energy_certificate = self.energy_certificate   #Energy certificate SEK/kWh
        self.model.compensation_fee = self.incentive_per_kwh    # Transfer compensation fee SEK/kWh
        self.model.resolution = self.resolution

        for charge_point in self.charging_points:
            setattr(self.model, f'{charge_point.name}_P_grid_buy', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(-1000000, 1000000)))
            setattr(self.model, f'{charge_point.name}_P_grid_sell', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(-1000000, 1000000)))
            setattr(self.model, f'{charge_point.name}_P_lec_buy', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(-1000000, 1000000)))
            setattr(self.model, f'{charge_point.name}_P_lec_sell', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(-1000000, 1000000)))
            setattr(self.model, f'{charge_point.name}_B_grid', pyo.Var(self.model.T, within=pyo.Binary))
            setattr(self.model, f'{charge_point.name}_B_lec', pyo.Var(self.model.T, within=pyo.Binary))

            # Iterate through each EV at the charging point
            for ev_index in range(len(charge_point.session_id)):
                ev_name = f'{charge_point.name}_S{charge_point.session_id[ev_index]}'  # Unique name for each EV
                setattr(self.model, f'{ev_name}_ch', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, charge_point.ev_max_power[ev_index])))
                setattr(self.model, f'{ev_name}_ds', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, self.v2g_on * charge_point.ev_max_power[ev_index])))
                setattr(self.model, f'{ev_name}_soc', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, 1)))
                setattr(self.model, f'{ev_name}_Bch', pyo.Var(self.model.T, within=pyo.Binary))

                ev_capacity = charge_point.ev_capacity[ev_index]
                ev_arrival_soc = charge_point.ev_arrival_soc[ev_index]
                ev_arrival = charge_point.ev_arrival[ev_index]
                ev_departure = charge_point.ev_departure[ev_index]
                ev_desired_soc = charge_point.ev_desired_soc[ev_index]

                def ev_soc_rule(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_ch = getattr(model, f'{ev_name}_ch')[t]
                    ev_ds = getattr(model, f'{ev_name}_ds')[t]
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    if t == ev_arrival:
                        return ev_soc == ev_arrival_soc + (ev_ch * charge_point.efficiency - ev_ds / charge_point.efficiency) / ev_capacity / self.resolution
                    elif ev_arrival < t <= ev_departure:
                        ev_previous_soc = getattr(model, f'{ev_name}_soc')[t - 1]
                        return ev_soc == ev_previous_soc + (ev_ch * charge_point.efficiency - ev_ds / charge_point.efficiency) / ev_capacity / self.resolution
                    else:
                        return pyo.Constraint.Skip
                setattr(self.model, f'{ev_name}_soc_constraint', pyo.Constraint(self.model.T, rule=ev_soc_rule))

                def ev_soc_min_rule(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    ev_arrival = charge_point.ev_arrival[ev_index]
                    ev_departure = charge_point.ev_departure[ev_index]
                    if ev_arrival <= t <= ev_departure:
                        return ev_soc >= 0.2
                    return ev_soc == 0
                setattr(self.model, f'{ev_name}_soc_min_constraint', pyo.Constraint(self.model.T, rule=ev_soc_min_rule))

                def ev_max_ch(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_Bch = getattr(model, f'{ev_name}_Bch')[t]
                    ev_ch = getattr(model, f'{ev_name}_ch')[t]
                    return ev_ch <= ev_Bch * charge_point.ev_max_power[ev_index]
                setattr(self.model, f'{ev_name}_max_ch_constraint', pyo.Constraint(self.model.T, rule=ev_max_ch))

                def ev_max_ds(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_Bch = getattr(model, f'{ev_name}_Bch')[t]
                    ev_ds = getattr(model, f'{ev_name}_ds')[t]
                    return ev_ds <= (1 - ev_Bch) * charge_point.ev_max_power[ev_index]
                setattr(self.model, f'{ev_name}_max_ds_constraint', pyo.Constraint(self.model.T, rule=ev_max_ds))

                def ev_avail_ch(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_ch = getattr(model, f'{ev_name}_ch')[t]
                    if charge_point.ev_arrival[ev_index] <= t < charge_point.ev_departure[ev_index]:
                        return ev_ch >= 0
                    return ev_ch == 0
                setattr(self.model, f'{ev_name}_avail_ch_constraint', pyo.Constraint(self.model.T, rule=ev_avail_ch))

                def ev_avail_ds(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_ds = getattr(model, f'{ev_name}_ds')[t]
                    if charge_point.ev_arrival[ev_index] <= t < charge_point.ev_departure[ev_index]:
                        return ev_ds >= 0
                    return ev_ds == 0
                setattr(self.model, f'{ev_name}_avail_ds_constraint', pyo.Constraint(self.model.T, rule=ev_avail_ds))

                def ev_desired_soc(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    if t == charge_point.ev_departure[ev_index]:
                        return ev_soc >= charge_point.ev_desired_soc[ev_index]
                    return pyo.Constraint.Skip
                setattr(self.model, f'{ev_name}_desired_soc_constraint', pyo.Constraint(self.model.T, rule=ev_desired_soc))

            def consumption(model, t, charge_point=charge_point):
                P_grid = getattr(model, f'{charge_point.name}_P_grid_buy')[t] - getattr(model, f'{charge_point.name}_P_grid_sell')[t]
                P_lec = getattr(model, f'{charge_point.name}_P_lec_buy')[t] - getattr(model, f'{charge_point.name}_P_lec_sell')[t]
                ev_power = sum(getattr(model, f'{charge_point.name}_S{charge_point.session_id[ev_index]}_ch')[t] - getattr(model, f'{charge_point.name}_S{charge_point.session_id[ev_index]}_ds')[t] for ev_index in range(charge_point.num_evs))
                return  ev_power == P_grid + P_lec
            setattr(self.model, f'{charge_point.name}_consumption_constraint', pyo.Constraint(self.model.T, rule=consumption))

            def max_import_cp(model, t, charge_point = charge_point):
                P_grid = getattr(model, f'{charge_point.name}_P_grid_buy')[t] #- getattr(model, f'{charge_point.name}_P_grid_sell')[t]
                P_lec = getattr(model, f'{charge_point.name}_P_lec_buy')[t] #- getattr(model, f'{charge_point.name}_P_lec_sell')[t]
                return P_grid + P_lec <= charge_point.ch_max_power
            setattr(self.model, f'{charge_point.name}_max_import_constraint', pyo.Constraint(self.model.T, rule = max_import_cp))

            def max_export_cp(model, t, charge_point = charge_point):
                P_grid = getattr(model, f'{charge_point.name}_P_grid_sell')[t]
                P_lec = getattr(model, f'{charge_point.name}_P_lec_sell')[t]
                return P_grid + P_lec <= charge_point.ds_max_power
            setattr(self.model, f'{charge_point.name}_max_export_constraint', pyo.Constraint(self.model.T, rule = max_export_cp))

            def cp_grid_exchange1(model, t, charge_point = charge_point):
                P_grid = getattr(model, f'{charge_point.name}_P_grid_buy')[t]
                B_grid = getattr(model, f'{charge_point.name}_B_grid')[t]
                return P_grid <= 100000 * B_grid
            setattr(self.model, f'{charge_point.name}_grid_exchange1', pyo.Constraint(self.model.T, rule = cp_grid_exchange1))

            def cp_grid_exchange2(model, t, charge_point = charge_point):
                P_grid = getattr(model, f'{charge_point.name}_P_grid_sell')[t]
                B_grid = getattr(model, f'{charge_point.name}_B_grid')[t]
                return P_grid <= 100000 * (1 - B_grid)
            setattr(self.model, f'{charge_point.name}_grid_exchange2', pyo.Constraint(self.model.T, rule = cp_grid_exchange2))

            def cp_lec_exchange1(model, t, charge_point = charge_point):
                P_lec = getattr(model, f'{charge_point.name}_P_lec_buy')[t]
                B_lec = getattr(model, f'{charge_point.name}_B_lec')[t]
                return P_lec <= 100000 * B_lec
            setattr(self.model, f'{charge_point.name}_lec_exchange1', pyo.Constraint(self.model.T, rule = cp_lec_exchange1))

            def cp_lec_exchange2(model, t, charge_point = charge_point):
                P_lec = getattr(model, f'{charge_point.name}_P_lec_sell')[t]
                B_lec = getattr(model, f'{charge_point.name}_B_lec')[t]
                return P_lec <= 100000 * (1 - B_lec)
            setattr(self.model, f'{charge_point.name}_lec_exchange2', pyo.Constraint(self.model.T, rule = cp_lec_exchange2))

            def cp_market_manipulation1(model, t, charge_point = charge_point):
                P_lec_sell = getattr(model, f'{charge_point.name}_P_lec_sell')[t]
                B_grid = getattr(model, f'{charge_point.name}_B_grid')[t]
                return P_lec_sell <= (1 - B_grid) * 10000
            setattr(self.model, f'{charge_point.name}_market_manipulation_constraint1', pyo.Constraint(self.model.T, rule = cp_market_manipulation1))

            def cp_market_manipulation2(model, t, charge_point = charge_point):
                P_grid_sell = getattr(model, f'{charge_point.name}_P_grid_sell')[t]
                B_lec = getattr(model, f'{charge_point.name}_B_lec')[t]
                return P_grid_sell <= (1 - B_lec) * 10000
            setattr(self.model, f'{charge_point.name}_market_manipulation_constraint2', pyo.Constraint(self.model.T, rule = cp_market_manipulation2))

        #Building constraints:
        for building in self.buildings:
            setattr(self.model, f'{building.name}_slack_imp', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(-1000000, 1000000), initialize = 0))
            setattr(self.model, f'{building.name}_P_grid_buy', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(-1000000, 1000000), initialize = 0))
            setattr(self.model, f'{building.name}_P_grid_sell', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(-1000000, 1000000), initialize = 0))
            setattr(self.model, f'{building.name}_P_lec_buy', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, building.el_int_ex_max), initialize = 0))
            setattr(self.model, f'{building.name}_P_lec_sell', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, building.el_int_ex_max), initialize = 0))
            setattr(self.model, f'{building.name}_B_lec', pyo.Var(self.model.T, within=pyo.Binary))
            setattr(self.model, f'{building.name}_B_grid', pyo.Var(self.model.T, within=pyo.Binary))
            setattr(self.model, f'{building.name}_B_el', pyo.Var(self.model.T, within=pyo.Binary))
            setattr(self.model, f'{building.name}_bess_soc', pyo.Var(self.model.T, within=pyo.Reals, bounds=(0, 1)))
            setattr(self.model, f'{building.name}_bess_ch', pyo.Var(self.model.T, within=pyo.Reals, bounds=(0, building.bess_max_power)))
            setattr(self.model, f'{building.name}_bess_ds', pyo.Var(self.model.T, within=pyo.Reals, bounds=(0, building.bess_max_power)))
            setattr(self.model, f'{building.name}_bess_Bch', pyo.Var(self.model.T, within=pyo.Binary))
            setattr(self.model, f'{building.name}_hp_P', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds = (0, building.hp_max_power)))
            setattr(self.model, f'{building.name}_hp_heat', pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0, building.hp_cop * 100000)))
            setattr(self.model, f'{building.name}_dh_heat', pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0,1000)))
            setattr(self.model, f'{building.name}_ch_P', pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0, building.ch_max_power)))
            setattr(self.model, f'{building.name}_ch_cool', pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0, building.ch_cop * 100000)))
            setattr(self.model, f'{building.name}_ch_heat_rec', pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0, building.ch_cop * 100000)))
            setattr(self.model, f'{building.name}_ch_heat_rec_spill', pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0, building.ch_cop * 100000)))
            setattr(self.model, f'{building.name}_dc_cool', pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0, building.dc_imp_max)))
            setattr(self.model, f'{building.name}_abs_ch_heat', pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0, building.abs_max_heat)))
            setattr(self.model, f'{building.name}_abs_ch_cool', pyo.Var(self.model.T, within = pyo.NonNegativeReals, bounds = (0, building.abs_ch_cop * 1000000)))
            setattr(self.model, f'{building.name}_heat_lec_buy', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds = (0, building.heat_int_ex_max)))
            setattr(self.model, f'{building.name}_heat_lec_sell', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds = (0, building.heat_int_ex_max)))
            setattr(self.model, f'{building.name}_B_heat_lec', pyo.Var(self.model.T, within=pyo.Binary))
            setattr(self.model, f'{building.name}_cool_lec_buy', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds = (0, building.cool_int_ex_max)))
            setattr(self.model, f'{building.name}_cool_lec_sell', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds = (0, building.cool_int_ex_max)))
            setattr(self.model, f'{building.name}_B_cool_lec', pyo.Var(self.model.T, within=pyo.Binary))

            def bess_soc_rule(model, t, building = building):
                bess_ch = getattr(model, f'{building.name}_bess_ch')[t]
                bess_ds = getattr(model, f'{building.name}_bess_ds')[t]
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                if building.bess_capacity == 0:
                    return bess_soc == 0
                elif t == 0:
                    return bess_soc == building.bess_initial_soc + (bess_ch*building.efficiency - bess_ds/building.efficiency)/building.bess_capacity
                else:
                    bess_previous_soc = getattr(model, f'{building.name}_bess_soc')[t-1]
                    return bess_soc == bess_previous_soc + (bess_ch*building.efficiency - bess_ds/building.efficiency)/building.bess_capacity
            setattr(self.model, f'{building.name}_bess_soc_constraint', pyo.Constraint(self.model.T, rule = bess_soc_rule))

            def bess_soc_min_rule(model, t, building = building):
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                if building.bess_capacity == 0:
                    return bess_soc == 0
                return bess_soc >= 0.2
            setattr(self.model, f'{building.name}_bess_soc_min_constraint', pyo.Constraint(self.model.T, rule = bess_soc_min_rule))  

            def bess_max_ch(model, t, building = building):
                bess_Bch = getattr(model, f'{building.name}_bess_Bch')[t]
                bess_ch = getattr(model, f'{building.name}_bess_ch')[t]
                if building.bess_capacity == 0:
                    return bess_ch == 0
                return bess_ch <= bess_Bch*building.bess_max_power
            setattr(self.model, f'{building.name}_bess_max_ch_constraint', pyo.Constraint(self.model.T, rule = bess_max_ch))

            def bess_max_ds(model, t, building = building):
                bess_Bch = getattr(model, f'{building.name}_bess_Bch')[t]
                bess_ds = getattr(model, f'{building.name}_bess_ds')[t]
                if building.bess_capacity == 0:
                    return bess_ds == 0
                return bess_ds <= (1-bess_Bch)*building.bess_max_power
            setattr(self.model, f'{building.name}_bess_max_ds_constraint', pyo.Constraint(self.model.T, rule = bess_max_ds))

            def hp_P_rule(model, t, building = building):
                hp_P = getattr(model, f'{building.name}_hp_P')[t]
                if building.hp_cop == 0:
                    return hp_P == 0
                return hp_P >= 0
            setattr(self.model, f'{building.name}_hp_P_constraint', pyo.Constraint(self.model.T, rule = hp_P_rule))

            def hp_heat_rule(model, t, building = building):
                hp_P = getattr(model, f'{building.name}_hp_P')[t]
                hp_heat = getattr(model, f'{building.name}_hp_heat')[t]
                return hp_heat == hp_P * building.hp_cop
            setattr(self.model, f'{building.name}_hp_heat_constraint', pyo.Constraint(self.model.T, rule = hp_heat_rule))

            def heating_balance_rule(model, t, building = building):
                hp_heat = getattr(model, f'{building.name}_hp_heat')[t]
                dh_heat = getattr(model, f'{building.name}_dh_heat')[t]
                ch_heat_rec = getattr(model, f'{building.name}_ch_heat_rec')[t]
                ch_heat_rec_spill = getattr(model, f'{building.name}_ch_heat_rec_spill')[t]
                abs_ch_heat = getattr(model, f'{building.name}_abs_ch_heat')[t]
                heat_lec_buy = getattr(model, f'{building.name}_heat_lec_buy')[t]
                heat_lec_sell = getattr(model, f'{building.name}_heat_lec_sell')[t]
                return heat_lec_buy - heat_lec_sell + hp_heat + dh_heat + ch_heat_rec - ch_heat_rec_spill == building.heating_load[t] + abs_ch_heat
            setattr(self.model, f'{building.name}_heating_balance_constraint', pyo.Constraint(self.model.T, rule = heating_balance_rule))

            def heating_lec_buy_rule(model, t, building = building):
                heat_lec_buy = getattr(model, f'{building.name}_heat_lec_buy')[t]
                B_heat_lec = getattr(model, f'{building.name}_B_heat_lec')[t]
                return heat_lec_buy <= B_heat_lec * 10000
            setattr(self.model, f'{building.name}_heating_lec_buy_constraint', pyo.Constraint(self.model.T, rule = heating_lec_buy_rule))

            def heating_lec_sell_rule(model, t, building = building):
                heat_lec_sell = getattr(model, f'{building.name}_heat_lec_sell')[t]
                B_heat_lec = getattr(model, f'{building.name}_B_heat_lec')[t]
                return heat_lec_sell <= (1 - B_heat_lec) * 10000
            setattr(self.model, f'{building.name}_heating_lec_sell_constraint', pyo.Constraint(self.model.T, rule = heating_lec_sell_rule))

            def heating_market_manipulation_rule(model, t, building = building):
                B_heat_lec = getattr(model, f'{building.name}_B_heat_lec')[t]
                dh_heat = getattr(model, f'{building.name}_dh_heat')[t]
                return dh_heat <= (B_heat_lec) * 10000  
            setattr(self.model, f'{building.name}_heating_market_manipulation_constraint', pyo.Constraint(self.model.T, rule = heating_market_manipulation_rule))

            def heating_recovery_spill_rule(model, t, building = building):
                ch_heat_rec = getattr(model, f'{building.name}_ch_heat_rec')[t]
                ch_heat_rec_spill = getattr(model, f'{building.name}_ch_heat_rec_spill')[t]
                return ch_heat_rec_spill <= ch_heat_rec
            setattr(self.model, f'{building.name}_heat_recovery_spill_constraint', pyo.Constraint(self.model.T, rule = heating_recovery_spill_rule))

            def ch_P_rule(model, t, building = building):
                ch_P = getattr(model, f'{building.name}_ch_P')[t]
                if building.ch_cop == 0:
                    return ch_P == 0
                return ch_P >= 0
            setattr(self.model, f'{building.name}_ch_p_constraint', pyo.Constraint(self.model.T, rule = ch_P_rule))

            def ch_cool_rule(model, t, building = building):
                ch_P = getattr(model, f'{building.name}_ch_P')[t]
                ch_cool = getattr(model, f'{building.name}_ch_cool')[t]
                return ch_cool == ch_P * building.ch_cop
            setattr(self.model, f'{building.name}_ch_cool_constraint', pyo.Constraint(self.model.T, rule = ch_cool_rule))

            def cooling_balance_rule(model, t, building = building):
                ch_cool = getattr(model, f'{building.name}_ch_cool')[t]
                dc_cool = getattr(model, f'{building.name}_dc_cool')[t]
                abs_ch_cool = getattr(model, f'{building.name}_abs_ch_cool')[t]
                cool_lec_buy = getattr(model, f'{building.name}_cool_lec_buy')[t]
                cool_lec_sell = getattr(model, f'{building.name}_cool_lec_sell')[t]
                return ch_cool + dc_cool + abs_ch_cool + cool_lec_buy - cool_lec_sell == building.cooling_load[t]
            setattr(self.model, f'{building.name}_cooling_balance_constraint', pyo.Constraint(self.model.T, rule = cooling_balance_rule))

            def cooling_lec_buy_rule(model, t, building = building):
                cool_lec_buy = getattr(model, f'{building.name}_cool_lec_buy')[t]
                B_cool_lec = getattr(model, f'{building.name}_B_cool_lec')[t]
                return cool_lec_buy <= B_cool_lec * 10000
            setattr(self.model, f'{building.name}_cooling_lec_buy_constraint', pyo.Constraint(self.model.T, rule = cooling_lec_buy_rule))

            def cooling_lec_sell_rule(model, t, building = building):
                cool_lec_sell = getattr(model, f'{building.name}_cool_lec_sell')[t]
                B_cool_lec = getattr(model, f'{building.name}_B_cool_lec')[t]
                return cool_lec_sell <= (1 - B_cool_lec) * 10000
            setattr(self.model, f'{building.name}_cooling_lec_sell_constraint', pyo.Constraint(self.model.T, rule = cooling_lec_sell_rule))

            def cooling_market_manipulation_rule(model, t, building = building):
                B_cool_lec = getattr(model, f'{building.name}_B_cool_lec')[t]
                dc_cool = getattr(model, f'{building.name}_dc_cool')[t]
                return dc_cool <= (B_cool_lec) * 10000  
            setattr(self.model, f'{building.name}_cooling_market_manipulation_constraint', pyo.Constraint(self.model.T, rule = cooling_market_manipulation_rule))

            def heat_recovery_chiller_rule(model, t, building = building):
                ch_cool = getattr(model, f'{building.name}_ch_cool')[t]
                ch_heat_rec = getattr(model, f'{building.name}_ch_heat_rec')[t]
                return ch_heat_rec == ch_cool * building.ch_heat_rec_eff
            setattr(self.model, f'{building.name}_heat_rec_chiller_constraint', pyo.Constraint(self.model.T, rule = heat_recovery_chiller_rule))

            def abs_chiller_rule(model, t, building = building):
                abs_ch_cool = getattr(model, f'{building.name}_abs_ch_cool')[t]
                abs_ch_heat = getattr(model, f'{building.name}_abs_ch_heat')[t]
                return abs_ch_heat == abs_ch_cool * building.abs_ch_cop
            setattr(self.model, f'{building.name}_abs_chiller_constraint', pyo.Constraint(self.model.T, rule = abs_chiller_rule))

            def building_consumption(model, t, building = building):
                #P = getattr(model, f'{building.name}_P_imp')[t] - getattr(model, f'{building.name}_P_exp')[t] + getattr(model, f'{building.name}_slack_imp')[t]
                P_slack = getattr(model, f'{building.name}_slack_imp')[t]
                P_grid = getattr(model, f'{building.name}_P_grid_buy')[t] - getattr(model, f'{building.name}_P_grid_sell')[t]
                P_lec = getattr(model, f'{building.name}_P_lec_buy')[t] - getattr(model, f'{building.name}_P_lec_sell')[t]
                hp_P = getattr(model, f'{building.name}_hp_P')[t]
                ch_P = getattr(model, f'{building.name}_ch_P')[t]
                load = building.load[t]
                pv = building.pv_production[t]
                bess_ch = getattr(model, f'{building.name}_bess_ch')[t]
                bess_ds = getattr(model, f'{building.name}_bess_ds')[t]
                return load - pv - bess_ds + bess_ch + hp_P + ch_P == P_grid + P_lec + P_slack
            setattr(self.model, f'{building.name}_consumption_constraint', pyo.Constraint(self.model.T, rule = building_consumption))

            def building_imp_max(model, t, building = building):
                P_grid = getattr(model, f'{building.name}_P_grid_buy')[t]
                P_lec = getattr(model, f'{building.name}_P_lec_buy')[t]
                P_imp = P_grid + P_lec
                B_el = getattr(model, f'{building.name}_B_el')[t]
                return P_imp <= B_el * building.el_imp_max
            setattr(self.model, f'{building.name}_imp_max_constraint', pyo.Constraint(self.model.T, rule = building_imp_max))

            def building_exp_max(model, t, building = building):
                P_grid = getattr(model, f'{building.name}_P_grid_sell')[t]
                P_lec = getattr(model, f'{building.name}_P_lec_sell')[t]
                P_exp = P_grid + P_lec
                B_el = getattr(model, f'{building.name}_B_el')[t]
                return P_exp <= (1 - B_el) * building.el_exp_max
            setattr(self.model, f'{building.name}_exp_max_constraint', pyo.Constraint(self.model.T, rule = building_exp_max))

            def building_grid_exchange1(model, t, building = building):
                P_grid_buy = getattr(model, f'{building.name}_P_grid_buy')[t]
                P_grid_sell = getattr(model, f'{building.name}_P_grid_sell')[t]
                B_grid = getattr(model, f'{building.name}_B_grid')[t]
                return P_grid_buy <= B_grid * 10000
            setattr(self.model, f'{building.name}_grid_exchange_constraint1', pyo.Constraint(self.model.T, rule = building_grid_exchange1))

            def building_grid_exchange2(model, t, building = building):
                P_grid_buy = getattr(model, f'{building.name}_P_grid_buy')[t]
                P_grid_sell = getattr(model, f'{building.name}_P_grid_sell')[t]
                B_grid = getattr(model, f'{building.name}_B_grid')[t]
                return P_grid_sell <= (1 - B_grid) * 10000
            setattr(self.model, f'{building.name}_grid_exchange_constraint2', pyo.Constraint(self.model.T, rule = building_grid_exchange2))

            def building_lec_exchange1(model, t, building = building):
                P_lec_buy = getattr(model, f'{building.name}_P_lec_buy')[t]
                P_lec_sell = getattr(model, f'{building.name}_P_lec_sell')[t]
                B_lec = getattr(model, f'{building.name}_B_lec')[t]
                return P_lec_buy <= B_lec * 10000
            setattr(self.model, f'{building.name}_lec_exchange_constraint1', pyo.Constraint(self.model.T, rule = building_lec_exchange1))

            def building_lec_exchange2(model, t, building = building):
                P_lec_buy = getattr(model, f'{building.name}_P_lec_buy')[t]
                P_lec_sell = getattr(model, f'{building.name}_P_lec_sell')[t]
                B_lec = getattr(model, f'{building.name}_B_lec')[t]
                return P_lec_sell <= (1 - B_lec) * 10000
            setattr(self.model, f'{building.name}_lec_exchange_constraint2', pyo.Constraint(self.model.T, rule = building_lec_exchange2))

            def stop_market_manipulation1(model, t, building = building):
                P_grid_buy = getattr(model, f'{building.name}_P_grid_buy')[t]
                P_lec_sell = getattr(model, f'{building.name}_P_lec_sell')[t]
                B_grid = getattr(model, f'{building.name}_B_grid')[t]
                B_lec = getattr(model, f'{building.name}_B_lec')[t]
                return P_lec_sell <= (1 - B_grid) * 10000
            setattr(self.model, f'{building.name}_market_manipulation_constraint1', pyo.Constraint(self.model.T, rule = stop_market_manipulation1))

            def stop_market_manipulation2(model, t, building = building):
                P_grid_sell = getattr(model, f'{building.name}_P_grid_sell')[t]
                P_lec_buy = getattr(model, f'{building.name}_P_lec_buy')[t]
                B_lec = getattr(model, f'{building.name}_B_lec')[t]
                return P_grid_sell <= (1 - B_lec) * 10000
            setattr(self.model, f'{building.name}_market_manipulation_constraint2', pyo.Constraint(self.model.T, rule = stop_market_manipulation2))
        
        def community_balance(model, t):
            return sum(getattr(model, f'{building.name}_P_lec_buy')[t] - getattr(model, f'{building.name}_P_lec_sell')[t] for building in self.buildings) + \
                    sum(getattr(model, f'{charge_point.name}_P_lec_buy')[t] - getattr(model, f'{charge_point.name}_P_lec_sell')[t] for charge_point in self.charging_points)== 0
        self.model.community_balance_constraint = pyo.Constraint(self.model.T, rule = community_balance)

        def community_heat_balance(model, t):
            return sum(getattr(model, f'{building.name}_heat_lec_buy')[t] - getattr(model, f'{building.name}_heat_lec_sell')[t] for building in self.buildings) == 0
        self.model.community_heat_balance_constraint = pyo.Constraint(self.model.T, rule = community_heat_balance)

        def community_cool_balance(model, t):
            return sum(getattr(model, f'{building.name}_cool_lec_buy')[t] - getattr(model, f'{building.name}_cool_lec_sell')[t] for building in self.buildings) == 0
        self.model.community_cool_balance_constraint = pyo.Constraint(self.model.T, rule = community_cool_balance)

        def power_balance(model, t):
            overall_consumption = sum(getattr(model, f'{charge_point.name}_P_grid_buy')[t] - getattr(model, f'{charge_point.name}_P_grid_sell')[t] for charge_point in self.charging_points) + \
                                    sum(getattr(model, f'{building.name}_P_grid_buy')[t] - getattr(model, f'{building.name}_P_grid_sell')[t] for building in self.buildings)
            P_im = self.model.P_im_grid[t]
            P_ex = self.model.P_ex_grid[t]
            return P_im - P_ex == overall_consumption
        self.model.power_balance_constarint = pyo.Constraint(self.model.T, rule=power_balance)

        def power_import(model, t):
            P_im = self.model.P_im_grid[t]
            B_im = self.model.B_im_grid[t]
            return P_im <= self.M * B_im
        self.model.power_import_constraint = pyo.Constraint(self.model.T, rule=power_import)

        def power_export(model, t):
            P_ex = self.model.P_ex_grid[t]
            B_im = self.model.B_im_grid[t]
            return P_ex <= self.M * (1 - B_im)
        self.model.power_export_constraint = pyo.Constraint(self.model.T, rule=power_export)

        def peak_load_constraint(model, t):
            return model.Peakload >= model.P_im_grid[t] - model.P_ex_grid[t]
        self.model.peak_load_cons = pyo.Constraint(self.model.T, rule=peak_load_constraint)

        def previous_peak_check1(model):
            return model.monthly_peak >= model.Peakload
        self.model.previous_peak_check1_constraint = pyo.Constraint(rule=previous_peak_check1)

        def previous_peak_check2(model):
            return model.monthly_peak >= model.previous_monthly_peak
        self.model.previous_peak_check2_constraint = pyo.Constraint(rule=previous_peak_check2)

        def peak_load_constraint_dh(model, t):
            return model.Peakload_dh >= self.model.dh_import[t]
        self.model.peak_load_cons_dh = pyo.Constraint(self.model.T, rule = peak_load_constraint_dh)

        def previous_peak_check1_dh(model):
            return model.monthly_peak_dh >= model.Peakload_dh
        self.model.previous_peak_check1_constraint_dh = pyo.Constraint(rule = previous_peak_check1_dh)

        def previous_peak_check2_dh(model):
            return model.monthly_peak_dh >= model.previous_monthly_peak_dh
        self.model.previous_peak_check2_constraint_dh = pyo.Constraint(rule = previous_peak_check2_dh)

        def peak_load_constraint_dc(model, t):
            return model.Peakload_dc >= self.model.dc_import[t]
        self.model.peak_load_cons_dc = pyo.Constraint(self.model.T, rule = peak_load_constraint_dc)

        def previous_peak_check1_dc(model):
            return model.monthly_peak_dc >= model.Peakload_dc
        self.model.previous_peak_check1_constraint_dc = pyo.Constraint(rule = previous_peak_check1_dc)

        def previous_peak_check2_dc(model):
            return model.monthly_peak_dc >= model.previous_monthly_peak_dc
        self.model.previous_peak_check2_constraint_dc = pyo.Constraint(rule = previous_peak_check2_dc)

        def dh_import_total(model, t):
            dh_import = self.model.dh_import[t]
            return dh_import == sum(getattr(model, f'{building.name}_dh_heat')[t] for building in self.buildings)
        self.model.dh_import_constraint = pyo.Constraint(self.model.T, rule = dh_import_total)

        def dc_import_total(model, t):
            dc_import = self.model.dc_import[t]
            return dc_import == sum(getattr(model, f'{building.name}_dc_cool')[t] for building in self.buildings)
        self.model.dc_import_constraint = pyo.Constraint(self.model.T, rule = dc_import_total)

        def tranmission_cost(model, t):
            return model.transmission_cost[t] == (model.P_im_grid[t] * model.Transmission_fee - model.P_ex_grid[t] * model.Transmission_health_incentive) / self.resolution
        self.model.tranmission_cost_constraint = pyo.Constraint(self.model.T, rule = tranmission_cost)

        def supplier_cost(model, t):
            return model.supplier_cost[t] == (model.P_im_grid[t] * (model.spot_prices[t] + model.Energy_certificate) - model.P_ex_grid[t] * (model.spot_prices[t] + model.Energy_certificate + model.compensation_fee)) / self.resolution
        self.model.supplier_cost_constraint = pyo.Constraint(self.model.T, rule = supplier_cost)

        def dh_cost_cal(model, t):
            return model.dh_cost[t] == model.dh_import[t] * model.dh_prices[t]
        self.model.dh_cost_constraint = pyo.Constraint(self.model.T, rule = dh_cost_cal)

        def dc_cost_cal(model, t):
            return model.dc_cost[t] == model.dc_import[t] * model.dc_prices[t]
        self.model.dc_cost_constraint = pyo.Constraint(self.model.T, rule = dc_cost_cal)

        def objective_rule(model):
            subscription_fee = model.Subscription_fee
            supplier_cost = sum(model.supplier_cost[t] for t in model.T)
            transmission_cost = sum(model.transmission_cost[t] for t in model.T)
            peak_cost = model.Effect_fee * model.monthly_peak * 5 #Check with "day/month" instead of 5
            peak_cost_dc = model.Effect_fee_dc * model.monthly_peak_dc * 5
            peak_cost_dh = model.Effect_fee_dh * model.monthly_peak_dh * 5
            dso_cost = transmission_cost + peak_cost + subscription_fee
            dh_cost = sum(model.dh_cost[t] for t in model.T) + model.Subscription_fee_dh
            dc_cost = sum(model.dc_cost[t] for t in model.T) + model.Subscription_fee_dc
            tax_cost = (supplier_cost + dso_cost + dh_cost + dc_cost)*0.25 + (1.25 * model.Energy_tax * sum(model.P_im_grid[t] - model.P_ex_grid[t] for t in model.T) / 4)
            slack_penalty = sum(getattr(model, f'{building.name}_slack_imp')[t] for building in self.buildings for t in model.T)
            overall_cost = dso_cost + tax_cost + supplier_cost + dh_cost + dc_cost + slack_penalty*10000000000 + peak_cost_dc + peak_cost_dh
            return overall_cost
        self.model.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

        def objective_rule_dc(model):
            ev_soc = sum(t * (getattr(model, f'{charge_point.name}_S{charge_point.session_id[ev_index]}_soc')[t]) for charge_point in self.charging_points for ev_index in range(charge_point.num_evs) for t in model.T)
            subscription_fee = model.Subscription_fee
            supplier_cost = sum(model.supplier_cost[t] for t in model.T)
            transmission_cost = sum(model.transmission_cost[t] for t in model.T)
            peak_cost = model.Effect_fee * model.monthly_peak * 5 #Check with "day/month" instead of 5
            peak_cost_dc = model.Effect_fee_dc * model.monthly_peak_dc * 5
            peak_cost_dh = model.Effect_fee_dh * model.monthly_peak_dh * 5
            dso_cost = transmission_cost + peak_cost + subscription_fee
            dh_cost = sum(model.dh_cost[t] for t in model.T) + model.Subscription_fee_dh
            dc_cost = sum(model.dc_cost[t] for t in model.T) + model.Subscription_fee_dc
            tax_cost = (supplier_cost + dso_cost + dh_cost + dc_cost)*0.25 + (1.25 * model.Energy_tax * sum(model.P_im_grid[t] - model.P_ex_grid[t] for t in model.T) / 4)
            overall_cost = dso_cost + tax_cost + supplier_cost + dh_cost + dc_cost - ev_soc * 1000 + peak_cost_dc + peak_cost_dh
            return overall_cost
        self.model.obj_dc = pyo.Objective(rule=objective_rule_dc, sense = pyo.minimize)

    def solve(self):
        solver = pyo.SolverFactory('gurobi')
        if self.dc:
            self.model.obj.deactivate()
            self.model.obj_dc.activate()
        else:
            self.model.obj.activate()
            self.model.obj_dc.deactivate()
        self.results = solver.solve(self.model)
        return self.results

    def get_results(self):
        print(f'Objective value: {pyo.value(self.model.obj)}')
        print(f"⏱ Time steps in model: {len(self.model.T)}")
        results = {}
        for cp in self.charging_points:
            T = list(self.model.T)
            ch = []
            ds = []
            soc = []
            ev_connection = []
            session_connection = []

            for t in T:
                ch_t = 0.0
                ds_t = 0.0
                connected_evs_t = None
                soc_ev_t = 0.0
                session_id_t = None

                for ev in range(cp.num_evs):
                    ev_name = f"{cp.name}_S{cp.session_id[ev]}"

                    ch_t += pyo.value(getattr(self.model, f"{ev_name}_ch")[t])
                    ds_t += pyo.value(getattr(self.model, f"{ev_name}_ds")[t])

                    soc_ev = pyo.value(getattr(self.model, f"{ev_name}_soc")[t])
                    soc_ev_t += soc_ev

                    if cp.ev_arrival[ev] <= t < cp.ev_departure[ev]:
                        connected_evs_t = cp.ev_id[ev]
                        session_id_t = cp.session_id[ev]
                
                ev_connection.append(connected_evs_t)
                session_connection.append(session_id_t)

                ch.append(ch_t)
                ds.append(ds_t)
                soc.append(soc_ev_t)

            # ---------- STORE CP-LEVEL RESULTS ----------
            results[f"{cp.name}_EV_connection"] = ev_connection
            results[f"{cp.name}_Session"] = session_connection
            results[f"{cp.name}_ch"] = ch
            results[f"{cp.name}_ds"] = ds
            results[f"{cp.name}_soc"] = soc

            # ---------- EXISTING CP VARIABLES ----------
            results[f"{cp.name}_P_grid_buy"] = [pyo.value(getattr(self.model, f"{cp.name}_P_grid_buy")[t]) for t in T]
            results[f"{cp.name}_P_grid_sell"] = [pyo.value(getattr(self.model, f"{cp.name}_P_grid_sell")[t]) for t in T]
            results[f"{cp.name}_P_lec_buy"] = [pyo.value(getattr(self.model, f"{cp.name}_P_lec_buy")[t]) for t in T]
            results[f"{cp.name}_P_lec_sell"] = [pyo.value(getattr(self.model, f"{cp.name}_P_lec_sell")[t]) for t in T]

        for building in self.buildings:
            results[f'{building.name}_load'] = [building.load[t] for t in self.model.T]
            results[f'{building.name}_PV'] = [building.pv_production[t] for t in self.model.T]
            results[f'{building.name}_bess_ch'] = [pyo.value(getattr(self.model, f'{building.name}_bess_ch')[t]) for t in self.model.T]
            results[f'{building.name}_bess_ds'] = [pyo.value(getattr(self.model, f'{building.name}_bess_ds')[t]) for t in self.model.T]
            results[f'{building.name}_bess_soc'] = [pyo.value(getattr(self.model, f'{building.name}_bess_soc')[t]) for t in self.model.T]
            results[f'{building.name}_el_lec_buy'] = [pyo.value(getattr(self.model, f'{building.name}_P_lec_buy')[t]) for t in self.model.T]
            results[f'{building.name}_el_lec_sell'] = [pyo.value(getattr(self.model, f'{building.name}_P_lec_sell')[t]) for t in self.model.T]
            results[f'{building.name}_el_grid_buy'] = [pyo.value(getattr(self.model, f'{building.name}_P_grid_buy')[t]) for t in self.model.T]
            results[f'{building.name}_el_grid_sell'] = [pyo.value(getattr(self.model, f'{building.name}_P_grid_sell')[t]) for t in self.model.T]
            results[f'{building.name}_slack_imp'] = [pyo.value(getattr(self.model, f'{building.name}_slack_imp')[t]) for t in self.model.T]
            results[f'{building.name}_hp_P'] = [pyo.value(getattr(self.model, f'{building.name}_hp_P')[t]) for t in self.model.T]
            results[f'{building.name}_hp_heat'] = [pyo.value(getattr(self.model, f'{building.name}_hp_heat')[t]) for t in self.model.T]
            results[f'{building.name}_dh_heat'] = [pyo.value(getattr(self.model, f'{building.name}_dh_heat')[t]) for t in self.model.T]
            results[f'{building.name}_ch_heat_rec'] = [pyo.value(getattr(self.model, f'{building.name}_ch_heat_rec')[t]) for t in self.model.T]
            results[f'{building.name}_ch_heat_rec_spill'] = [pyo.value(getattr(self.model, f'{building.name}_ch_heat_rec_spill')[t]) for t in self.model.T]
            results[f'{building.name}_abs_ch_heat'] = [pyo.value(getattr(self.model, f'{building.name}_abs_ch_heat')[t]) for t in self.model.T]
            results[f'{building.name}_heat_lec_buy'] = [pyo.value(getattr(self.model, f'{building.name}_heat_lec_buy')[t]) for t in self.model.T]
            results[f'{building.name}_heat_lec_sell'] = [pyo.value(getattr(self.model, f'{building.name}_heat_lec_sell')[t]) for t in self.model.T]
            results[f'{building.name}_load_heat'] = [building.heating_load[t] for t in self.model.T]
            results[f'{building.name}_ch_P'] = [pyo.value(getattr(self.model, f'{building.name}_ch_P')[t]) for t in self.model.T]
            results[f'{building.name}_ch_cool'] = [pyo.value(getattr(self.model, f'{building.name}_ch_cool')[t]) for t in self.model.T]
            results[f'{building.name}_dc_cool'] = [pyo.value(getattr(self.model, f'{building.name}_dc_cool')[t]) for t in self.model.T]
            results[f'{building.name}_abs_ch_cool'] = [pyo.value(getattr(self.model, f'{building.name}_abs_ch_cool')[t]) for t in self.model.T]
            results[f'{building.name}_cool_lec_buy'] = [pyo.value(getattr(self.model, f'{building.name}_cool_lec_buy')[t]) for t in self.model.T]
            results[f'{building.name}_cool_lec_sell'] = [pyo.value(getattr(self.model, f'{building.name}_cool_lec_sell')[t]) for t in self.model.T]
            results[f'{building.name}_load_cool'] = [building.cooling_load[t] for t in self.model.T]
        results['P_import'] = [pyo.value(self.model.P_im_grid[t]) for t in self.model.T]
        results['P_export'] = [pyo.value(self.model.P_ex_grid[t]) for t in self.model.T]
        results['dh_import'] = [pyo.value(self.model.dh_import[t]) for t in self.model.T]
        results['dc_import'] = [pyo.value(self.model.dc_import[t]) for t in self.model.T]
        results['Transmission cost'] = [pyo.value(self.model.transmission_cost[t]) for t in self.model.T]
        results['Supplier cost'] = [pyo.value(self.model.supplier_cost[t]) for t in self.model.T]
        results['DH cost'] = [pyo.value(self.model.dh_cost[t]) for t in self.model.T]
        results['DC cost'] = [pyo.value(self.model.dc_cost[t]) for t in self.model.T]
        return pd.DataFrame(results)

In [3]:
#name, ev_capacity, ev_max_power, ev_arrival_soc, ev_arrival, ev_departure, ev_desired_soc
T = 24

# Spot prices (€/kWh)
spot_prices = [0.05 + 0.01*np.sin(i*np.pi/12) for i in range(T)]
dh_prices = [0.0 + 0.01*np.sin(i*np.pi/12) for i in range(T)]
dc_prices = [1.75 + 0.01*np.sin(i*np.pi/12) for i in range(T)]

# 24-hour realistic load and PV
load = [15 + 0.5*np.sin(i*np.pi/12) for i in range(T)]
pv_production = [0.0 if i < 6 or i > 18 else 1.5*np.sin((i-6)*np.pi/12) for i in range(T)]
load1 = [10 + 0.5*np.sin(i*np.pi/12) for i in range(T)]
pv_production1 = [0.0 if i < 6 or i > 18 else 2.5*np.sin((i-6)*np.pi/12) for i in range(T)]
heating_profile = [5 + 2*np.sin(i*np.pi/12) for i in range(T)]
cooling_profile = [2.5 + 2*np.sin(i*np.pi/12) for i in range(T)]

cp1 = charging_point(name = 'cp1', ev_id = ['sq', 1], session_id= [1, 2], ev_capacity=[45, 65], ev_max_power=[10, 12], ev_arrival=[6, 18], ev_departure= [12, 23], ev_arrival_soc=[0.5, 0.3], ev_desired_soc=[0.75, 0.4])
cp2 = charging_point(name = 'cp2', ev_id = [3, 2], session_id= [1, 2], ev_capacity=[55, 95], ev_max_power=[10, 12], ev_arrival=[8, 15], ev_departure= [12, 20], ev_arrival_soc=[0.6, 0.7], ev_desired_soc=[0.75, 0.75])

b1 = building(name= 'b1', load = load, pv_production=pv_production, bess_capacity=0, bess_initial_soc=0.75, bess_max_power=15, heating_load=heating_profile, has_hp=True, hp_cop=3, hp_max_power=20,
              cooling_load=cooling_profile, has_ch = True, ch_cop = 2, ch_max_power = 3, ch_heat_rec_eff=0.1, has_abs_ch= False, abs_ch_cop=0.8, abs_max_heat=10,
              el_exp_max=30, el_imp_max=120)
b2 = building(name= 'b2', load = load1, pv_production=pv_production1, bess_capacity=800, bess_initial_soc=0.95, bess_max_power=75, heating_load=heating_profile, has_hp=False, hp_cop=3, hp_max_power=2,
              cooling_load=cooling_profile, has_ch = True, ch_cop = 2, ch_max_power = 3, ch_heat_rec_eff=0.1, has_abs_ch= False, abs_ch_cop=0.8, abs_max_heat=10,
              el_exp_max=30, el_imp_max=120)
b3 = building(name= 'b3', load = [0*i for i in range(len(load1))], pv_production=[0*i for i in range(len(pv_production1))], bess_capacity=80, bess_initial_soc=0.5, bess_max_power=7.5)

opt_model = LEC_Opt_spot_multi_energy([], [b1, b2], spot_prices, dh_prices, dc_prices=dh_prices,dc = False, peak_effect_fee_dc=2, peak_effect_fee_dh=1)
results_df = opt_model.solve()
df = opt_model.get_results()
df

Objective value: 819.7566787584983
⏱ Time steps in model: 24


,b1_load,b1_PV,b1_bess_ch,b1_bess_ds,b1_bess_soc,b1_el_lec_buy,b1_el_lec_sell,b1_el_grid_buy,b1_el_grid_sell,b1_slack_imp,...,b2_cool_lec_sell,b2_load_cool,P_import,P_export,dh_import,dc_import,Transmission cost,Supplier cost,DH cost,DC cost
0,15.000000,0.000000e+00,0.0,0.0,0.0,12.459541,0.0,3.388988,0.0,0.0,...,1.500000,2.500000,3.388988,0.000000e+00,7.054416,1.0,0.382956,0.186394,0.000000,0.000000
1,15.129410,0.000000e+00,0.0,0.0,0.0,13.908352,0.0,3.388988,0.0,0.0,...,0.000000,3.017638,3.388988,0.000000e+00,7.054416,1.0,0.382956,0.195166,0.018258,0.002588
2,15.250000,0.000000e+00,0.0,0.0,0.0,15.059541,0.0,3.388988,0.0,0.0,...,0.000000,3.500000,3.388988,0.000000e+00,7.054416,1.0,0.382956,0.203339,0.035272,0.005000
3,15.353553,0.000000e+00,0.0,0.0,0.0,15.618729,0.0,3.388988,0.0,0.0,...,0.000000,3.914214,3.388988,0.000000e+00,7.054416,1.0,0.382956,0.210358,0.049882,0.007071
4,15.433013,0.000000e+00,0.0,0.0,0.0,14.663835,0.0,3.388988,0.0,0.0,...,1.767949,4.232051,3.388988,0.000000e+00,7.054416,1.0,0.382956,0.215744,0.061093,0.008660
5,15.482963,0.000000e+00,0.0,0.0,0.0,15.817540,0.0,3.388988,0.0,0.0,...,0.000000,4.431852,3.388988,0.000000e+00,7.054416,1.0,0.382956,0.219129,0.068140,0.009659
6,15.500000,0.000000e+00,0.0,0.0,0.0,15.909541,0.0,3.388988,0.0,0.0,...,0.000000,4.500000,3.388988,0.000000e+00,7.054416,1.0,0.382956,0.220284,0.070544,0.010000
7,15.482963,3.882286e-01,0.0,0.0,0.0,15.429312,0.0,3.388988,0.0,0.0,...,0.000000,4.431852,3.388988,0.000000e+00,7.054416,1.0,0.382956,0.219129,0.068140,0.009659
8,15.433013,7.500000e-01,0.0,0.0,0.0,13.913835,0.0,3.388988,0.0,0.0,...,1.767949,4.232051,3.388988,0.000000e+00,7.054416,1.0,0.382956,0.215744,0.061093,0.008660
9,15.353553,1.060660e+00,0.0,0.0,0.0,14.558069,0.0,3.388988,0.0,0.0,...,0.000000,3.914214,3.388988,0.000000e+00,7.054416,1.0,0.382956,0.210358,0.049882,0.007071


In [6]:
try:
    base_dir = Path(__file__).parent  # works when running as a script
except NameError:
    base_dir = Path.cwd()  # fallback for Jupyter or interactive mode

charging_point_file_path = base_dir / "Data" / "CP_sessions_2023.xlsx"
charging_point_data = pd.read_excel(charging_point_file_path, sheet_name=None)
building_file_path = base_dir / "Data" / f"building_data_multi_energy.xlsx"
building_data = pd.read_excel(building_file_path, index_col = [0], sheet_name=None)

prices_file_path = base_dir / "Data" / "prices_2023.xlsx"
prices = pd.read_excel(prices_file_path)
prices.index = pd.DatetimeIndex(prices.iloc[:,0])

start_date = pd.to_datetime("2023-01-01 00:00:00")
subscription_fee = 1100/30
subscription_fee_dh = 117820/365
subscription_fee_dc = 80020/365
transmission_fee = 0.055
transmission_health_incentive = 0
peak_effect_fee = 57.9/30
peak_effect_fee_dh = 1185/365
peak_effect_fee_dc = 695/365
energy_tax = 0.439
energy_certificate = 0.005
incentive_per_kwh=0
days = 1
horizon_hours = 36
store_hours = 24
previous_monthly_peak = 0
previous_monthly_peak_heat = 0
previous_monthly_peak_cool = 0
current_month=1
previous_soc = pd.DataFrame(index=range(1), columns=list(building_data.keys()))
previous_soc.iloc[:, :] = 0.5
building_on = 1
charging_point_on = 1
unfeasible_days = []
# Store rolling results
rolling_results = pd.DataFrame()

max_cool = 0
for i in building_data.keys():
    max_cool = max(building_data[f'{i}']['cooling_load'].max(), max_cool)
    if max_cool > 1000:
        peak_effect_fee_dc = 586/365
        subscription_fee_dc = 149020/365

# Track ongoing EV sessions across days
if charging_point_on == 1:
    ongoing_sessions = {cp_name: [] for cp_name in charging_point_data.keys()}

for day in range(days):
    print("=" * 40)
    print(f"🔄 Day {day + 1} Optimization ({horizon_hours}h Horizon)")

    resolution = int(3600/(prices['Spot prices'].index[1] - prices['Spot prices'].index[0]).total_seconds())

    if resolution == 1:
        freq_index = '60min'
    elif resolution == 4:
        freq_index = '15min'
    else:
        print('Resolution of spot prices is not 15 min or 60 min!! - CHECK')
        break
    
    start_time = start_date + timedelta(days=day)
    end_time = start_time + timedelta(hours=horizon_hours) - pd.Timedelta(seconds=1)
    opt_end_time = start_time + timedelta(hours=store_hours) - pd.Timedelta(seconds=1)
    full_index = pd.date_range(start=start_time,end=end_time,freq=f'{freq_index}')
    
    # 1. Build buildings
    print("Step 1: Initializing buildings")
    building_list = []
    if building_on == 1:
        for name in building_data.keys():
            
            load = building_data[name].loc[start_time:end_time - timedelta(hours=1), 'electricity_load']
            load = load[~load.index.duplicated(keep='first')]
            load = load.reindex(full_index, method = 'ffill') / resolution

            pv = building_data[name].loc[start_time:end_time - timedelta(hours=1), 'pv_production']
            pv = pv[~pv.index.duplicated(keep='first')]
            pv = pv.reindex(full_index, method = 'ffill') / resolution

            bess_capacity = building_data[name]['bess_capacity'].iloc[0]
            bess_max_power = building_data[name]['bess_power'].iloc[0]

            cooling_load = building_data[name].loc[start_time:end_time - timedelta(hours=1), 'cooling_load']
            cooling_load = cooling_load[~cooling_load.index.duplicated(keep='first')]
            cooling_load = cooling_load.reindex(full_index, method = 'ffill') / resolution

            heating_load = building_data[name].loc[start_time:end_time - timedelta(hours=1), 'heating_load']
            heating_load = heating_load[~heating_load.index.duplicated(keep='first')]
            heating_load = heating_load.reindex(full_index, method = 'ffill') / resolution

            if building_data[name]['has_hp'].iloc[0] == 1:
                has_hp = True
            else:
                has_hp = False
            hp_cop = building_data[name]['hp_cop'].iloc[0]
            hp_max_power = building_data[name]['hp_max_power'].iloc[0]

            if building_data[name]['has_ch'].iloc[0] == 1:
                has_ch = True
                ch_heat_recovery_eff = building_data[name]['ch_heat_rec_eff'].iloc[0]  
            else:
                has_ch = False
                ch_heat_recovery_eff = 0 
            ch_cop = building_data[name]['ch_cop'].iloc[0]
            ch_max_power = building_data[name]['ch_max_power'].iloc[0] 

            el_imp_max = building_data[name]['el_imp_max'].iloc[0]         
            el_exp_max = building_data[name]['el_exp_max'].iloc[0]    
            el_int_max = building_data[name]['el_int_max'].iloc[0]    
            heat_int_max = building_data[name]['heat_int_max'].iloc[0] 
            cool_int_max = building_data[name]['cool_int_max'].iloc[0]    

            dc_imp_max = building_data[name]['dc_imp_max'].iloc[0]


            print(f"  - Building {name} with initial SOC {previous_soc[name].iloc[0]}")
            building_list.append(building(
                name=name,
                load=load,
                pv_production=pv,
                bess_capacity=bess_capacity,
                bess_initial_soc=previous_soc[name].iloc[0],
                bess_max_power=bess_max_power,
                heating_load=heating_load,
                has_hp=has_hp,
                hp_cop=hp_cop,
                hp_max_power=hp_max_power,
                cooling_load=cooling_load,
                has_ch=has_ch,
                ch_cop=ch_cop,
                ch_max_power=ch_max_power,
                el_exp_max=el_exp_max,
                el_imp_max=el_imp_max,
                el_int_ex_max=el_int_max,
                heat_int_ex_max=heat_int_max,
                cool_int_ex_max=cool_int_max,
                ch_heat_rec_eff = ch_heat_recovery_eff,
                dc_imp_max = dc_imp_max
            ))
    else:
        print('Skipped buildings!!!')
    
    # 2. Charging Points
    print("Step 2: Preparing charging points")
    charging_point_list = []
    if charging_point_on == 1:
        new_ongoing_sessions = {cp_name: [] for cp_name in charging_point_data.keys()}
        new_sessions_starting_tomorrow = []

        for cp_name in charging_point_data.keys():
            print(f"  ⛽ Charging Point: {cp_name}")
            cp_df = charging_point_data[cp_name]
            cp_df['Arrival'] = pd.to_datetime(cp_df['Arrival'])
            cp_df['Departure'] = pd.to_datetime(cp_df['Departure'])
            capacities, max_powers, arrivals, departures = [], [], [], []
            arrival_socs, desired_socs, ev_ids, session_ids = [], [], [], []

            # a) Add ongoing sessions
            print("   ↪ Checking ongoing sessions from previous day")
            for idx, session in enumerate(ongoing_sessions[cp_name]):
                if session['departure_time'] > start_time:
                    dep_time = int((session['departure_time'] - start_time).total_seconds() // (60 * 60 / resolution))
                    ev_index = len(capacities)
                    session['ev_index'] = ev_index
                    capacities.append(session['capacity'])
                    max_powers.append(session['max_power'])
                    arrivals.append(0)
                    departures.append(min(dep_time, horizon_hours * 4))
                    arrival_socs.append(session['last_soc'])
                    if session['departure_time'] < end_time:
                        desired_socs.append(session['last_desired_soc'])
                    else:
                        desired_socs.append(session['last_desired_soc'])
                    ev_ids.append(session['ev_id'])
                    session_ids.append(session['session_id'])

                    print(f"     ✅ Continued EV{session['ev_id']}: dep_time_slot={dep_time}, SOC={session['last_soc']}")
                    if session['departure_time'] > opt_end_time:
                        new_ongoing_sessions[cp_name].append(session)

            # b) Add new sessions that start in first 24 hours
            print("   ↪ Adding new sessions starting in first 24 hours")
            today_sessions = cp_df[
                (cp_df['Arrival'] >= start_time) &
                (cp_df['Arrival'] < start_time + timedelta(hours=store_hours))
            ]

            for idx, row in today_sessions.iterrows():
                arrival_time = int((row['Arrival'] - start_time).total_seconds() // (60 * 60 / resolution))
                departure_time = int((row['Departure'] - start_time).total_seconds() // (60 * 60 / resolution))

                capacities.append(row['Capacity'])
                max_powers.append(row['Max_Power'])
                arrivals.append(arrival_time)

                if row['Departure'] > end_time:
                    connected_time = departure_time - arrival_time
                    slope_linear = (row['Desired SOC'] - row['Arrival SOC']) / connected_time
                    linear_requested = slope_linear * (horizon_hours * resolution - arrival_time)
                    departures.append(horizon_hours * resolution) #need to fix if they connect just before the end of horizon then what should be desired soc?
                    session = {
                        'capacity': row['Capacity'],
                        'max_power': row['Max_Power'],
                        'departure_time': row['Departure'],
                        'last_soc': None,
                        'desired_soc': row['Arrival SOC'] + linear_requested, #row['Desired SOC'],
                        'last_desired_soc': row['Desired SOC'],
                        'cp_name': cp_name,
                        'ev_index': len(capacities) - 1,
                        'ev_id': row['ev_id'],
                        'session_id': row['session_id']
                    }
                    new_ongoing_sessions[cp_name].append(session)
                    new_sessions_starting_tomorrow.append(session)
                    print(f"     ➕ New EV (spans days): arrival {arrival_time}, will depart next day and can end day with SOC: {row['Arrival SOC'] + linear_requested}")
                    
                elif row['Departure'] >= opt_end_time:
                    departures.append(departure_time)
                    session = {
                        'capacity': row['Capacity'],
                        'max_power': row['Max_Power'],
                        'departure_time': row['Departure'],
                        'last_soc': None,
                        'desired_soc': row['Desired SOC'],
                        'last_desired_soc': row['Desired SOC'],
                        'cp_name': cp_name,
                        'ev_index': len(capacities) - 1,
                        'ev_id': row['ev_id'],
                        'session_id': row['session_id']
                    }
                    print(f"     ➕ New EV (spans days): arrival {arrival_time}, will depart next day at: {departure_time}")
                    new_ongoing_sessions[cp_name].append(session)
                    new_sessions_starting_tomorrow.append(session)
                else:
                    departures.append(departure_time)
                    print(f"     ➕ New EV: arrival {arrival_time}, departure {departure_time}")

                arrival_socs.append(row['Arrival SOC'])
                desired_socs.append(row['Desired SOC'])
                ev_ids.append(row['ev_id'])
                session_ids.append(row['session_id'])

            charging_point_list.append(charging_point(
                name=cp_name,
                ev_id = ev_ids,
                session_id= session_ids,
                ev_capacity=capacities,
                ev_max_power=max_powers,
                ev_arrival=arrivals,
                ev_departure=departures,
                ev_arrival_soc=arrival_socs,
                ev_desired_soc=desired_socs
            ))
        else:
            print('Skipped charging_points!!')

    # 3. Run optimization
    print("Step 3: Solving optimization model")
    spot_prices = np.array(prices['Spot prices'].loc[start_time:end_time])
    dh_prices = np.array(prices['DH'].loc[start_time:end_time])
    dc_prices = np.array(prices['DC'].loc[start_time:end_time])
    
    opt_model = LEC_Opt_spot_multi_energy(charging_point_list, building_list, spot_prices, resolution=resolution ,v2g_on=1, dc = False, subscription_fee = subscription_fee, transmission_fee = transmission_fee,
                transmission_health_incentive = transmission_health_incentive, peak_effect_fee = peak_effect_fee, energy_tax = energy_tax, energy_certificate = energy_certificate, incentive_per_kwh=incentive_per_kwh,
                dh_prices=dh_prices, dc_prices=dc_prices, peak_effect_fee_dh=peak_effect_fee_dh, peak_effect_fee_dc=peak_effect_fee_dc, subscription_fee_dc=subscription_fee_dc, subscription_fee_dh=subscription_fee_dh,
                previous_monthly_peak=previous_monthly_peak, previous_monthly_peak_dc=previous_monthly_peak_cool, previous_monthly_peak_dh=previous_monthly_peak_heat)
    results_df = opt_model.solve()
    if (results_df.solver.termination_condition == TerminationCondition.infeasible) or (results_df.solver.termination_condition == TerminationCondition.infeasibleOrUnbounded):
        print('Model is infeasible!!!!')
        break
    elif (results_df.solver.termination_condition == TerminationCondition.infeasible):
        print('Model is unbounded!! check the decision variable bounds!!')
        break
    elif (results_df.solver.termination_condition == TerminationCondition.optimal):
        print('Optimial solution found')
        df = opt_model.get_results()
        df.index = prices.loc[start_time:end_time, 'Spot prices'].index
    else:
        print(f'Solver stopped with condition: {results_df.solver.termination_condition}')
        break
    print("   ✅ Optimization complete")
    for col in df.columns:
        if '_soc' in col:
            print(col, "→", df[col].iloc[store_hours * resolution - 1])
    print("\n✅ Columns in results DataFrame:")
    print([col for col in df.columns if '_soc' in col])

    # Save first 24h of results
    print("Step 3: Saving 24h results to cumulative DataFrame")
    rolling_results = pd.concat([rolling_results, df.iloc[:store_hours * resolution]])
    opt_month = rolling_results.index[-1].month
    if current_month - opt_month == 0:
        opt_peak = (df['P_import'].iloc[:store_hours * resolution] - df['P_export'].iloc[:store_hours * resolution]).max()   #needs to be from the opt
        if previous_monthly_peak < opt_peak:
            previous_monthly_peak = opt_peak
        opt_peak_dh = (df['dh_import'].iloc[:store_hours * resolution]).max()
        if previous_monthly_peak_heat < opt_peak_dh:
            previous_monthly_peak_heat = opt_peak_dh
        opt_peak_dc = (df['dc_import'].iloc[:store_hours * resolution]).max()
        if previous_monthly_peak_cool < opt_peak_dc:
            previous_monthly_peak_cool = opt_peak_dc
    else:
        previous_monthly_peak = 0
        current_month = opt_month
        previous_monthly_peak_cool = 0
        previous_monthly_peak_heat = 0

    # 4. Update building SOCs
    if building_on == 1:
        print("Step 4: Updating building SOCs")
        for name in building_data.keys():
            soc_val = df[f'{name}_bess_soc'].iloc[store_hours * resolution - 1]
            previous_soc[name].iloc[0] = soc_val
            print(f"  🔋 {name} end-of-day SOC: {soc_val:.2f}")
    else:
        print('Step 4. No buildings available')

    if charging_point_on == 1:

        # 5. Update ongoing sessions' SOCs
        print("Step 5: Updating ongoing session SOCs from Day", day + 1)
        for cp_name in ongoing_sessions.keys():
            for session in ongoing_sessions[cp_name]:
                ev_name = f"{session['cp_name']}"
                if ev_name + "_soc" in df.columns:
                    session['last_soc'] = df[f'{ev_name}_soc'].iloc[store_hours * resolution - 1]
                    print(f"🔄 Updated {ev_name} SOC = {session['last_soc']:.2f}")
                else:
                    print(f"⚠️ Warning: {ev_name}_soc not found in results")

        # 6. Update new sessions for tomorrow with today’s end SOC
        print("Step 6: Updating sessions starting today that continue to tomorrow")
        for session in new_sessions_starting_tomorrow:
            ev_name = f"{session['cp_name']}"
            if f"{ev_name}_soc" in df.columns:
                session['last_soc'] = float(df[f"{ev_name}_soc"].iloc[store_hours * resolution - 1])
                print(f"🚚 {ev_name}: SOC carried to next day = {session['last_soc']:.2f}")
            else:
                print(f"⚠️ Could not update SOC for {ev_name}")

        # 7. Carry forward ongoing sessions
        print("Step 7: Updating ongoing_sessions for next day")
        ongoing_sessions = new_ongoing_sessions
    

    print("=" * 40 + "\n")
peak_load = (rolling_results['P_import']-rolling_results['P_export']).resample('M').max()
monthly_peak_costs = peak_load * peak_effect_fee / 24 / resolution
month_end_index = rolling_results.index.to_period('M').to_timestamp('M')
rolling_results['Peak cost'] = month_end_index.map(monthly_peak_costs)
rolling_results['DSO cost'] = rolling_results['Transmission cost'] + rolling_results['Peak cost'] + subscription_fee / 24 / resolution
rolling_results['DH peak cost'] = rolling_results['dh_import'].clip(lower=1750) * peak_effect_fee_dh / 24 / resolution
rolling_results['DH cost'] = rolling_results['DH peak cost'] + subscription_fee_dh / 24 / resolution + rolling_results['DH cost']
rolling_results['DC peak cost'] = rolling_results['dc_import'].clip(lower=280) * peak_effect_fee_dc / 24 / resolution
rolling_results['DC cost'] = rolling_results['DC peak cost'] + subscription_fee_dc / 24 / resolution + rolling_results['DC cost']
#rolling_results['Tax cost'] = 0.25 * (rolling_results['Supplier cost'] + rolling_results['DSO cost'] + rolling_results['DH cost'] + rolling_results['DC cost']) + 1.25 * energy_tax * (rolling_results['P_import'] - rolling_results['P_export']) / resolution
rolling_results['Tax cost'] = 1 * energy_tax * (rolling_results['P_import'] - rolling_results['P_export']) / resolution
rolling_results['Overall cost'] = rolling_results['DSO cost'] + rolling_results['Supplier cost'] + rolling_results['Tax cost'] + rolling_results['DH cost'] + rolling_results['DC cost']
rolling_results['El_int_exchange'] = rolling_results.filter(regex='_el_lec_buy').sum(axis=1)
rolling_results['Heat_int_exchange'] = rolling_results.filter(regex='_heat_lec_buy').sum(axis=1)
rolling_results['Cool_int_exchange'] = rolling_results.filter(regex='_cool_lec_buy').sum(axis=1)

🔄 Day 1 Optimization (36h Horizon)
Step 1: Initializing buildings
  - Building SCA with initial SOC 0.5
  - Building VB with initial SOC 0.5
  - Building SCAextra with initial SOC 0.5
  - Building ResBuilding with initial SOC 0.5
Step 2: Preparing charging points
  ⛽ Charging Point: CP1
   ↪ Checking ongoing sessions from previous day
   ↪ Adding new sessions starting in first 24 hours
     ➕ New EV (spans days): arrival 14, will depart next day and can end day with SOC: 0.5958556729552239
  ⛽ Charging Point: CP2
   ↪ Checking ongoing sessions from previous day
   ↪ Adding new sessions starting in first 24 hours
     ➕ New EV: arrival 39, departure 80
Skipped charging_points!!
Step 3: Solving optimization model
Optimial solution found
Objective value: 59487.9969247799
⏱ Time steps in model: 144
   ✅ Optimization complete
CP1_soc → 1.0
CP2_soc → 0.0
SCA_bess_soc → 0.912255018644303
VB_bess_soc → 0.0
SCAextra_bess_soc → 0.0
ResBuilding_bess_soc → 0.0

✅ Columns in results DataFrame:
['CP